# Homework: Vector Search

The homework is available
[here](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/02-vector-search/homework.md).

## Preparations

### Prepare Workspace

In [57]:
# dependencies
import numpy as np

from tqdm.auto import tqdm
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

from embedder import Embedder

## Exercises

### Exercise 1

In [2]:
# define stings or rather questions to embed
q1 = "How does approximate nearest neighbor search work?"

# initialize embedder - model from ONNX weights plus some surrounding code
embed = Embedder()

# embed the question
v1 = embed.encode(q1)

# question: What's the first value of the embedding (v[0])?
print(f"The first value of the embedding (v[0]) is: {np.round(v1[0], 2)}")

The first value of the embedding (v[0]) is: -0.02


### Exercise 2

In [3]:
# source course lessons
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
# get a specific document
doc1 = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

print(doc1["content"])

# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over every document, so it takes a minute on our dataset.
Keeping everything in memory is fine here, but a larger dataset would
need too much space.

The third problem is brute-force search. For every query we compare the
query vector against every single document. With 1,000 documents this is
fine, probably even faster than anything smarter. But as the dataset
grows past 10,000 or so, it gets slow, and we'll want an approximate
method instead.

What we've done so far is exact nearest neighbor (NN) sea

In [6]:
# embed document
vdoc1 = embed.encode(doc1["content"])

In [7]:
# question asks about cosine similarity
# of question and document
print(
    f"Cosine similarity between question and document: "
    f"{np.round(vdoc1.dot(np.round(v1, 2)), 2)}"
)

Cosine similarity between question and document: 0.36


### Exercise 3

In [9]:
# chunk documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [32]:
# embed in batches
batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [c["content"] for c in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [35]:
# compute dot product of query to each document
scores = X.dot(v1)

In [41]:
# get document with highest score
print(
    f"Index of document with highest score: {scores.argmax()}.\n"
    f"Score: {scores[scores.argmax()]}"
)

# get the document
chunks[scores.argmax()]

Index of document with highest score: 94.
Score: 0.648901732433228


{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

### Exercise 4

In [46]:
# check key of chunks to see what to index exactly
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [47]:
# initialize a vector search index
vindex = VectorSearch()

# fit the index to the documents
vindex.fit(X, chunks)

In [ ]:
# question to query the index on from homework exercise
q2 = "What metric do we use to evaluate a search engine?"

# embed the question
v2 = embed.encode(q2)

In [56]:
# query index on q2
vindex.search(v2)[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

### Exercise 5

In [61]:
# initialize a keyword search index
index = Index(text_fields=["content"])

# fit the index to the documents
index.fit(chunks)

In [62]:
# define new query
q3 = "How do I store vectors in PostgreSQL?"

# embed the question
v3 = embed.encode(q3)

In [70]:
# get filenames top 5 results of vector index
print("Vector index results:")
[c["filename"] for c in vindex.search(v3)[:5]]

Vector index results:


['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [71]:
# get filenames top 5 results of keyword index
print("Keyword index results:")
[c["filename"] for c in index.search(q3)[:5]]

Keyword index results:


['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

### Exercise 6

In [73]:
# define function for reciprocal rank fusion
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [87]:
# another question for querying the indices
q4 = "How do I give the model access to tools?"

# embed the question
v4 = embed.encode(q4)

In [88]:
# get results from vector and key search
vector_results = vindex.search(v4)
keyword_results = index.search(q4)

# apply reciprocal rank fusion
rre_results =rrf([vector_results, keyword_results])

# print vector results
print("Vector index results:")
for result in vector_results[:5]:
    print(result["filename"])

# print keyword results
print("\nKeyword index results:")
for result in keyword_results[:5]:
    print(result["filename"])

# get filenames of top 5 results
print("\nReciprocal Rank Fusion results:")
for result in rre_results[:5]:
    print(result["filename"])


Vector index results:
01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md

Keyword index results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md

Reciprocal Rank Fusion results:
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
